# Algoritmo de Bellman-Ford

Calcula distancias mínimas desde un origen admitiendo pesos **negativos**. Relaja todas las aristas $V - 1$ veces (ningún camino mínimo tiene más aristas); si en una pasada más todavía mejora alguna distancia, hay un ciclo de peso negativo. Cuesta $O(V \cdot E)$.

## 1. Distancias con pesos negativos

Calcular las distancias mínimas desde `A` en un grafo dado como lista de aristas `(origen, destino, peso)`, detectando ciclos negativos.

In [1]:
def bellman_ford(nodos, aristas, origen):
    dist = {n: float("inf") for n in nodos}
    dist[origen] = 0
    for _ in range(len(nodos) - 1):
        cambio = False
        for u, v, w in aristas:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                cambio = True
        # sin cambios: ya es definitivo
        if not cambio:
            break
    if any(dist[u] + w < dist[v] for u, v, w in aristas):
        return None
    return dist

nodos = "ABCDE"
aristas = [("A", "B", 6), ("A", "C", 7), ("B", "D", 5), ("B", "E", -4), ("C", "D", -3), ("C", "E", 9), ("D", "B", -2), ("E", "D", 7)]
print(bellman_ford(nodos, aristas, "A"))

{'A': 0, 'B': 2, 'C': 7, 'D': 4, 'E': -2}


## 2. Arbitraje de divisas

Con tipos de cambio $r_{ij}$, existe arbitraje si cambiando en ciclo se acaba con más dinero: $\prod r > 1 \iff \sum -\log r < 0$. Detectarlo buscando ciclos negativos con pesos $-\log r$.

In [2]:
from math import log

cambios = {
    ("EUR", "USD"): 1.08, ("USD", "EUR"): 0.93,
    ("USD", "JPY"): 150.0, ("JPY", "USD"): 0.0067,
    ("EUR", "JPY"): 158.0, ("JPY", "EUR"): 0.0064,
}

def hay_arbitraje(cambios):
    divisas = {d for par in cambios for d in par}
    aristas = [(a, b, -log(r)) for (a, b), r in cambios.items()]
    # origen ficticio conectado a todo para ver todos los ciclos
    aristas += [("*", d, 0) for d in divisas]
    return bellman_ford(divisas | {"*"}, aristas, "*") is None

print(hay_arbitraje(cambios))
print(1.08 * 150.0 * 0.0064)

True
1.0368


## 3. Encontrar el ciclo negativo

Guardar el predecesor de cada nodo y hacer $V$ pasadas. Si en la última todavía se relaja una arista hacia $v$, retroceder $V$ veces por los predecesores desde $v$ asegura caer dentro del ciclo, que se recorre hasta volver al mismo nodo. Mostrar la secuencia de cambios de divisas que da beneficio y cuánto se gana.

In [3]:
from math import prod

def ciclo_negativo(nodos, aristas):
    # empezar con todo a 0 equivale al origen ficticio unido a todos los nodos
    dist = {n: 0 for n in nodos}
    previo = {n: None for n in nodos}
    for _ in range(len(nodos)):
        ultimo = None
        for u, v, w in aristas:
            if dist[u] + w < dist[v]:
                dist[v], previo[v] = dist[u] + w, u
                ultimo = v
    if ultimo is None:
        return None
    for _ in range(len(nodos)):
        ultimo = previo[ultimo]
    ciclo = [ultimo]
    while len(ciclo) == 1 or ciclo[-1] != ultimo:
        ciclo.append(previo[ciclo[-1]])
    return ciclo[::-1]

divisas = sorted({d for par in cambios for d in par})
ciclo = ciclo_negativo(divisas, [(a, b, -log(r)) for (a, b), r in cambios.items()])
ganancia = prod(cambios[a, b] for a, b in zip(ciclo, ciclo[1:]))
assert ganancia > 1
print(" → ".join(ciclo), f"multiplica el dinero por {ganancia:.4f}")

print(ciclo_negativo(nodos, aristas))
con_ciclo = aristas + [("E", "C", -10)]
ciclo = ciclo_negativo(nodos, con_ciclo)
pesos = {(u, v): w for u, v, w in con_ciclo}
print(ciclo, "peso", sum(pesos[a, b] for a, b in zip(ciclo, ciclo[1:])))

JPY → USD → JPY multiplica el dinero por 1.0050
None
['E', 'C', 'D', 'B', 'E'] peso -19


## 4. Comprobación contra fuerza bruta

En grafos aleatorios pequeños con pesos negativos, comprobar que Bellman-Ford detecta un ciclo negativo alcanzable desde el origen exactamente cuando lo hay al probar todos los ciclos simples, y que, si no lo hay, cada distancia es el mínimo sobre todos los caminos simples.

In [4]:
import random
from itertools import permutations

def coste(camino, pesos):
    return sum(pesos[a, b] for a, b in zip(camino, camino[1:]))

def es_camino(camino, pesos):
    return all((a, b) in pesos for a, b in zip(camino, camino[1:]))

def distancias_fuerza_bruta(n, pesos, origen):
    dist = {v: float("inf") for v in range(n)}
    dist[origen] = 0
    otros = [v for v in range(n) if v != origen]
    for largo in range(1, n):
        for camino in permutations(otros, largo):
            camino = (origen,) + camino
            if es_camino(camino, pesos):
                dist[camino[-1]] = min(dist[camino[-1]], coste(camino, pesos))
    return dist

def ciclo_negativo_alcanzable(n, pesos, alcanzables):
    for largo in range(2, n + 1):
        for ciclo in permutations(range(n), largo):
            ciclo += (ciclo[0],)
            if ciclo[0] in alcanzables and es_camino(ciclo, pesos) and coste(ciclo, pesos) < 0:
                return True
    return False

random.seed(9)
negativos = 0
for _ in range(400):
    n = random.randint(1, 5)
    pesos = {(u, v): random.randint(-4, 8) for u in range(n) for v in range(n) if u != v and random.random() < 0.4}
    resultado = bellman_ford(range(n), [(u, v, w) for (u, v), w in pesos.items()], 0)
    fuerza_bruta = distancias_fuerza_bruta(n, pesos, 0)
    alcanzables = {v for v, d in fuerza_bruta.items() if d < float("inf")}
    hay_ciclo = ciclo_negativo_alcanzable(n, pesos, alcanzables)
    assert (resultado is None) == hay_ciclo
    if hay_ciclo:
        negativos += 1
    else:
        assert resultado == fuerza_bruta
print(negativos, "de 400 grafos con ciclo negativo alcanzable")

73 de 400 grafos con ciclo negativo alcanzable
